# Valorant YOLO Training
**Runtime → Change runtime type → T4 GPU** before running.

At the end, `models/valorant.pt` will be downloaded to your PC.

In [ ]:
# 1. Install dependencies
!pip install ultralytics roboflow -q

In [ ]:
# 2. Download dataset from Roboflow
from roboflow import Roboflow

API_KEY = "PASTE_YOUR_API_KEY_HERE"   # <-- paste your Roboflow API key

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("valo-qs4up").project("valorant-0h6ni")
dataset = project.version(1).download("yolov8")
print("Dataset path:", dataset.location)

In [ ]:
# 3. Verify GPU
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# 4. Train
from ultralytics import YOLO
import os

DATA_YAML = os.path.join(dataset.location, "data.yaml")

model = YOLO("yolov8m.pt")

results = model.train(
    data=DATA_YAML,
    epochs=40,
    batch=32,
    imgsz=640,
    patience=15,
    save=True,
    save_period=5,      # checkpoint every 5 epochs — safety net if Colab disconnects
    plots=True,
    augment=True,
    mixup=0.1,
    device=0,
    workers=4,
    cache=True,
    project="runs",
    name="valorant_v1",
)

print("Training complete!")

In [ ]:
# 5. Copy best weights and download to your PC
import shutil
from google.colab import files

best = "runs/valorant_v1/weights/best.pt"
shutil.copy(best, "valorant.pt")

files.download("valorant.pt")
print("Download started — save it to your models/ folder as valorant.pt")

In [ ]:
# Optional: view training curves
from IPython.display import Image
Image("runs/valorant_v1/results.png")